# Order Block

__Algorithm:__ impulsive-move detection → Order Block marking → higher-timeframe bias filter → confirmation → entry with SL/TP.

__Pattern:__
1. Mark an "Order Block" (OB): the last opposing candle immediately before an impulsive move.
2. Wait for price to revisit the OB zone and print a same-direction confirmation candle \
(engulfs the zone AND either EMA-fast crosses EMA-slow OR closes past the OB extreme).
3. Only take the trade when the higher-timeframe bias agrees.

__Features:__
- Fully automated OB detection (last opposing candle before strong impulse)
- Higher-timeframe (15m) EMA20 bias filter
- Simple but effective confirmation (engulfing + EMA9 cross)
- ATR-based impulse detection + consecutive candle filter

__How Order Block Algorithm Determines Entry/Exit:__
- Impulse: either a single candle with body > ob_body_mult * ATR, \
OR a run of >= ob_consecutive_min same-direction candles whose combined body exceeds ob_consecutive_range_pct of the latest close.
- Stop  = OB extreme ± (ob_stop_buffer_pct * extreme).
- Target = entry ± ob_rr * |entry − stop|.

## Configuration

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
from engine.fetcher import BybitFetcher
from engine.backtester import Backtester
from engine.models import StrategyConfig, Signal, SignalAction
from engine.visualization import build_chart
from IPython.display import HTML, display

In [ ]:
SYMBOL   = "BTCUSDT"
INTERVAL = "15"
CANDLES  = 800

In [ ]:
# Fetch data
fetcher = BybitFetcher()
df = fetcher.fetch_klines(
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=CANDLES,
    start_time="2026-03-20",
    end_time="2026-04-19",    # optional; defaults to now
)
fetcher.close()

## Order Block

In [ ]:
# Import Order Block strategy
from engine.strategies import OrderBlockStrategy


In [ ]:
# Backtest Order Block strategy
config = StrategyConfig()
strategy = OrderBlockStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

In [ ]:
# Dollar P&L
INITIAL_BALANCE = 100  # USD

balance = INITIAL_BALANCE
peak = balance
max_dd = 0

print(f"\n{'─' * 45}")
print(f"  Dollar P&L (starting ${INITIAL_BALANCE:,.2f})")
print(f"{'─' * 45}")

for i, t in enumerate(result.trades, 1):
    prev = balance
    balance *= (1 + t.pnl_bps / 10_000)
    peak = max(peak, balance)
    max_dd = max(max_dd, (peak - balance) / peak)
    pnl = balance - prev
    print(f"  #{i:3d}  {t.direction.value:5s}  {pnl:+8.2f}  →  ${balance:,.2f}")

print(f"{'─' * 45}")
print(f"  Final balance  : ${balance:,.2f}")
print(f"  Net profit     : ${balance - INITIAL_BALANCE:+,.2f}")
print(f"  Return         : {(balance / INITIAL_BALANCE - 1):+.2%}")
print(f"  Max drawdown   : {max_dd:.2%}")

In [ ]:
# Order Block strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

# build_chart saves to HTML — to display inline instead:
chart_file = f"chart_{strategy.name}.html"

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
    save_path=chart_file,
)

# Display inline
display(HTML(open(chart_file).read()))

## Inverse Order Block

Identical to OrderBlockStrategy but with flipped direction:
- Bullish OB confirmed → SHORT (fading the bounce)
- Bearish OB confirmed → LONG  (fading the rejection)

Stop and target distances are mirrored around entry, so the R:R geometry is preserved — only the direction flips. \
Useful as a mean-reversion counter-trend play against the standard OB.

In [ ]:
# Import inverse Order Block strategy
from engine.strategies import InverseOrderBlockStrategy

In [ ]:
# Backtest inverse Order Block strategy
config = StrategyConfig()
strategy = InverseOrderBlockStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

In [ ]:
# Dollar P&L
INITIAL_BALANCE = 100  # USD

balance = INITIAL_BALANCE
peak = balance
max_dd = 0

print(f"\n{'─' * 45}")
print(f"  Dollar P&L (starting ${INITIAL_BALANCE:,.2f})")
print(f"{'─' * 45}")

for i, t in enumerate(result.trades, 1):
    prev = balance
    balance *= (1 + t.pnl_bps / 10_000)
    peak = max(peak, balance)
    max_dd = max(max_dd, (peak - balance) / peak)
    pnl = balance - prev
    print(f"  #{i:3d}  {t.direction.value:5s}  {pnl:+8.2f}  →  ${balance:,.2f}")

print(f"{'─' * 45}")
print(f"  Final balance  : ${balance:,.2f}")
print(f"  Net profit     : ${balance - INITIAL_BALANCE:+,.2f}")
print(f"  Return         : {(balance / INITIAL_BALANCE - 1):+.2%}")
print(f"  Max drawdown   : {max_dd:.2%}")

In [ ]:
# Inverse Order Block strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

# build_chart saves to HTML — to display inline instead:
chart_file = f"chart_{strategy.name}.html"

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
    save_path=chart_file,
)

# Display inline
display(HTML(open(chart_file).read()))

## Live signals

Live mode:
- It generates signals, it does not place orders. \
There's no exchange API key, no order execution. It tells you when to enter/exit.
- State persists \
If you stop and restart, it remembers whether you're in a position via trading_state.db.
- Circuit breaker \
If Bybit is unreachable 10 times in a row, it stops automatically instead of spinning forever.
- Chart updates in place \
Automatic: the chart refreshes in the browser every poll_seconds.

To actually execute trades automatically, you need to add authenticated Bybit order placement on top of the signal output.

If you don't want to re-download automatically, two edits locally:
1. visualization.py — add auto_refresh: int = 0 parameter to build_chart, and after fig.write_html(save_path):
pythonif auto_refresh > 0:
    with open(save_path, "r") as f:
        html = f.read()
    meta_tag = f'<meta http-equiv="refresh" content="{auto_refresh}">'
    html = html.replace("<head>", f"<head>{meta_tag}", 1)
    with open(save_path, "w") as f:
        f.write(html)
2. live.py — add auto_refresh=self.poll_seconds to the build_chart() call in _tick().

### From CLI (command line interface)

- runs in a loop
- polls (refreshes) Bybit every 30 seconds
- persists state to SQLite (survives restarts)
- writes a chart to live_chart.html each tick
- handles SIGTERM/Ctrl+C gracefully

In [ ]:
python -m engine \
    --strategy order_block \
    --mode live \
    --symbol BTCUSDT \
    --interval 15 \
    --candles 500 \
    --poll 30 \
    --save live_chart.html \
    --db trading_state.db

### From a notebook cell

In [ ]:
from engine.live import LiveEngine
from engine.models import StrategyConfig
from engine.strategies import OrderBlockStrategy

SYMBOL   = "BTCUSDT"
INTERVAL = "15"

config = StrategyConfig()
strategy = OrderBlockStrategy(config)

engine = LiveEngine(
    strategy=strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=30,
    chart_path=f"live_{strategy.name}.html",
    db_path="trading_state.db",
)

engine.run()  # blocks until Ctrl+C or kernel interrupt